# Sentence Structure Evaluator

**The Sentence Structure Evaluator** measures the grammatical complexity of a text relative to its target grade, for students in grades 3-12. It runs a 2-stage pipeline:

1. **Sentence analysis** (`sentence_analysis` step) — an LLM performs a sentence-by-sentence grammatical analysis (sentence types, clauses, phrases, transitions), using deterministic ground-truth counts (sentence/word/char/syllable counts, Flesch-Kincaid grade) as a reference.
2. **Engineered features** — ~30 percentage/ratio features (sentence-type distribution, subordination density, phrase density, transitions, sentence-length buckets) are derived from stage 1's raw counts.
3. **Complexity classification** (`classify_complexity` step) — an LLM combines the engineered features with a grade-specific rubric (Grade 3 / Grade 4 / Grades 5-12 have distinct rubrics) to produce the final complexity rating.

The final output includes:

* **complexity_score**: `slightly_complex` to `exceedingly_complex`
* **reasoning**: how the qualitative structure and quantitative statistics combine to produce that rating

> **Known limitation carried over from the SDK, not introduced here:** `complexity-user.txt` instructs the model to answer with one of `["Slightly Complex", "Moderately Complex", "Very Complex", "Exceedingly Complex"]` (Title Case), while `output_schema.json` requires the snake_case enum used across every other evaluator in this repo (`slightly_complex`, etc.). Structured output below enforces the true (snake_case) contract regardless of that prose — left as-is per maintainer direction, same treatment as Vocabulary Complexity's analogous prompt/schema mismatch.

Everything this notebook runs — the models, the temperatures, the prompts, the output schema — is loaded from `config.json` in this directory. Nothing is hardcoded below, so the notebook cannot drift from the canonical assets.

In [ ]:
%pip install -qU langchain-openai langchain textstat python-dotenv

In [ ]:
import getpass
import hashlib
import json
import os
import pprint as pp
import statistics
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import textstat

In [ ]:
# Check for the API key
load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

### Load the config

`config.json` is the source of truth. Prompt files are loaded from disk and verified against the
`sha256` recorded for each step's messages.

> `input_schema.json` is a CI-side contract only (`scripts/check.py` binds each fixture's `input` to
> it) and isn't needed at runtime, so only `output_schema.json` is loaded here — it drives structured
> output for the `classify_complexity` step. The `sentence_analysis` step's own rich structured
> output (~30 grammatical counts) is an internal intermediate schema, defined below in code since
> `config.schema.json` only declares one evaluator-level `output_schema`.

In [ ]:
ASSETS_DIR = Path(".")


def load_config():
    """Load config.json + hash-verified prompts for every step + output schema."""
    with open(ASSETS_DIR / "config.json") as f:
        config = json.load(f)
    with open(ASSETS_DIR / "output_schema.json") as f:
        output_schema = json.load(f)

    steps = {}
    for step in config["steps"]:
        messages = []
        for msg_spec in step["prompt"]["messages"]:
            path = ASSETS_DIR / msg_spec["source_path"]
            text = path.read_text()
            actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
            declared_sha = msg_spec["sha256"]
            assert actual_sha == declared_sha, (
                f"prompt drift detected in step {step['id']!r} "
                f"role={msg_spec['role']!r} ({msg_spec['source_path']}): "
                f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
            )
            messages.append((msg_spec["role"], text))
        steps[step["id"]] = {"spec": step, "messages": messages}

    return config, output_schema, steps


CONFIG, OUTPUT_SCHEMA, STEPS = load_config()

print(f"Loaded {CONFIG['evaluator']['id']}")
for step_id, step in STEPS.items():
    model = step["spec"]["model"]
    sha_list = ", ".join(
        f"{role}={hashlib.sha256(text.encode()).hexdigest()[:12]}"
        for role, text in step["messages"]
    )
    print(f"    {step_id:<20} model={model['provider']}/{model['name']:<10} {sha_list}")

### Ground truth counts preprocessing

Declared as the `ground_truth_counts` preprocessing entry in `config.json`. These are deterministic
counts the `sentence_analysis` LLM step is told to use as a reference (not a hard constraint —
its own heuristics can override ambiguous cases).

In [ ]:
def compute_ground_truth_counts(text: str) -> str:
    """Per CONFIG['preprocessing'] entry 'ground_truth_counts'."""
    gt_sentence_count = textstat.sentence_count(text)
    gt_word_count = textstat.lexicon_count(text, removepunct=True)
    gt_char_count = textstat.char_count(text, ignore_spaces=True)
    gt_syllable_count = textstat.syllable_count(text)
    fk_grade = round(textstat.flesch_kincaid_grade(text), 2)
    return (
        f"num_sentences: {gt_sentence_count}\n"
        f"num_words: {gt_word_count}\n"
        f"num_char: {gt_char_count}\n"
        f"num_syllable: {gt_syllable_count}\n"
        f"flesch_kincaid_grade: {fk_grade}"
    )

### Stage 1: sentence analysis

The `sentence_analysis` step's own structured-output schema (~30 grammatical counts), matching the
TS SDK's `SentenceAnalysisSchema` field-for-field. This is an internal intermediate schema, not the
evaluator's final `output_schema.json`.

In [ ]:
SENTENCE_ANALYSIS_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "reasoning", "num_sentences", "num_words", "flesch_kincaid_grade",
        "num_simple_sentences", "num_compound_sentences", "num_complex_sentences",
        "num_compound_complex_sentences", "num_other_sentences",
        "num_independent_clauses", "num_subordinate_clauses", "num_total_clauses",
        "num_sentences_with_subordinate", "num_sentences_with_multiple_subordinates",
        "num_sentences_with_embedded_clauses", "num_prepositional_phrases",
        "num_participle_phrases", "num_appositive_phrases", "num_simple_transitions",
        "num_sophisticated_transitions", "words_in_simple_sentences",
        "words_in_compound_sentences", "words_in_complex_sentences",
        "words_in_compound_complex_sentences", "words_in_other_sentences",
        "sentence_word_counts", "num_one_concept_sentences", "num_multi_concept_sentences",
        "num_cleft_sentences", "max_clauses_in_any_sentence", "num_compound",
        "num_basic_complex", "num_advanced_complex", "percentage_simple",
        "percentage_compound", "percentage_basic_complex", "percentage_advanced_complex",
    ],
    "properties": {
        "reasoning": {"type": "string", "description": "Step-by-step reasoning for the analysis"},
        "num_sentences": {"type": "integer"},
        "num_words": {"type": "integer"},
        "flesch_kincaid_grade": {"type": "number"},
        "num_simple_sentences": {"type": "integer"},
        "num_compound_sentences": {"type": "integer"},
        "num_complex_sentences": {"type": "integer"},
        "num_compound_complex_sentences": {"type": "integer"},
        "num_other_sentences": {"type": "integer"},
        "num_independent_clauses": {"type": "integer"},
        "num_subordinate_clauses": {"type": "integer"},
        "num_total_clauses": {"type": "integer"},
        "num_sentences_with_subordinate": {"type": "integer"},
        "num_sentences_with_multiple_subordinates": {"type": "integer"},
        "num_sentences_with_embedded_clauses": {"type": "integer"},
        "num_prepositional_phrases": {"type": "integer"},
        "num_participle_phrases": {"type": "integer"},
        "num_appositive_phrases": {"type": "integer"},
        "num_simple_transitions": {"type": "integer"},
        "num_sophisticated_transitions": {"type": "integer"},
        "words_in_simple_sentences": {"type": "integer"},
        "words_in_compound_sentences": {"type": "integer"},
        "words_in_complex_sentences": {"type": "integer"},
        "words_in_compound_complex_sentences": {"type": "integer"},
        "words_in_other_sentences": {"type": "integer"},
        "sentence_word_counts": {"type": "array", "items": {"type": "integer"}},
        "num_one_concept_sentences": {"type": "integer"},
        "num_multi_concept_sentences": {"type": "integer"},
        "num_cleft_sentences": {"type": "integer"},
        "max_clauses_in_any_sentence": {"type": "integer"},
        "num_compound": {"type": "integer"},
        "num_basic_complex": {"type": "integer"},
        "num_advanced_complex": {"type": "integer"},
        "percentage_simple": {"type": "number"},
        "percentage_compound": {"type": "number"},
        "percentage_basic_complex": {"type": "number"},
        "percentage_advanced_complex": {"type": "number"},
    },
}


def run_sentence_analysis(text: str) -> dict:
    step = STEPS["sentence_analysis"]["spec"]
    llm = ChatOpenAI(model=step["model"]["name"], temperature=step["generation"]["temperature"])
    structured_llm = llm.with_structured_output(SENTENCE_ANALYSIS_SCHEMA, include_raw=True)

    ground_truth_counts = compute_ground_truth_counts(text)
    prompt_template = ChatPromptTemplate.from_messages(STEPS["sentence_analysis"]["messages"])
    rendered_messages = prompt_template.format_messages(text=text, ground_truth_counts=ground_truth_counts)
    raw = structured_llm.invoke(rendered_messages)

    if raw.get("parsing_error"):
        raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

    return {
        "rendered_prompt": [m.model_dump() for m in rendered_messages],
        "raw_output": raw["raw"],
        "raw_text": raw["raw"].content,
        "formatted_output": raw["parsed"],
        "usage": getattr(raw["raw"], "usage_metadata", None),
    }

### Engineered features

Derives the `sentence_features` preprocessing entry's ~30 percentage/ratio features from stage 1's
raw counts. Ported field-for-field from the TS SDK's `addEngineeredFeatures()`.

In [ ]:
def safe_div(numerator, denominator):
    return (numerator / denominator) if denominator else 0


def categorize_sentence_lengths(word_counts):
    if not word_counts:
        return {
            "percent_short_sentences": 0, "percent_medium_sentences": 0,
            "percent_long_sentences": 0, "percent_very_long_sentences": 0,
        }
    short = medium = long_ = very_long = 0
    for count in word_counts:
        if count <= 10:
            short += 1
        elif count <= 20:
            medium += 1
        elif count <= 30:
            long_ += 1
        else:
            very_long += 1
    total = len(word_counts)
    return {
        "percent_short_sentences": short / total * 100,
        "percent_medium_sentences": medium / total * 100,
        "percent_long_sentences": long_ / total * 100,
        "percent_very_long_sentences": very_long / total * 100,
    }


def add_engineered_features(a: dict) -> dict:
    """Per CONFIG['preprocessing'] entry 'sentence_features'."""
    f = dict(a)
    n_sentences = a["num_sentences"]
    n_words = a["num_words"]
    word_counts = a.get("sentence_word_counts") or []

    f["avg_words_per_sentence"] = safe_div(n_words, n_sentences)
    f["sentence_length_variation"] = statistics.pstdev(word_counts) if len(word_counts) > 1 else 0
    f.update(categorize_sentence_lengths(word_counts))

    f["percent_simple_sentences"] = safe_div(a["num_simple_sentences"], n_sentences) * 100
    f["percent_compound_sentences"] = safe_div(a["num_compound_sentences"], n_sentences) * 100
    f["percent_complex_sentences"] = safe_div(a["num_complex_sentences"], n_sentences) * 100
    f["percent_compound_complex_sentences"] = safe_div(a["num_compound_complex_sentences"], n_sentences) * 100
    f["percent_other_sentences"] = safe_div(a["num_other_sentences"], n_sentences) * 100

    f["percent_words_in_simple_sentences"] = safe_div(a["words_in_simple_sentences"], n_words) * 100
    f["percent_words_in_compound_sentences"] = safe_div(a["words_in_compound_sentences"], n_words) * 100
    f["percent_words_in_complex_sentences"] = safe_div(a["words_in_complex_sentences"], n_words) * 100
    f["percent_words_in_compound_complex_sentences"] = safe_div(a["words_in_compound_complex_sentences"], n_words) * 100
    f["percent_words_in_other_sentences"] = safe_div(a["words_in_other_sentences"], n_words) * 100

    f["avg_subordinates_per_sentence"] = safe_div(a["num_subordinate_clauses"], n_sentences)
    f["avg_clauses_per_sentence"] = safe_div(a["num_total_clauses"], n_sentences)
    f["percent_sentences_with_subordinate"] = safe_div(a["num_sentences_with_subordinate"], n_sentences) * 100
    f["percent_sentences_with_multiple_subordinates"] = safe_div(a["num_sentences_with_multiple_subordinates"], n_sentences) * 100
    f["percent_sentences_with_embedded_clauses"] = safe_div(a["num_sentences_with_embedded_clauses"], n_sentences) * 100

    f["prep_phrase_density"] = safe_div(a["num_prepositional_phrases"], n_words) * 100
    f["participle_phrase_density"] = safe_div(a["num_participle_phrases"], n_words) * 100
    f["appositive_phrase_density"] = safe_div(a["num_appositive_phrases"], n_words) * 100

    total_transitions = a["num_simple_transitions"] + a["num_sophisticated_transitions"]
    f["avg_transitions_per_sentence"] = safe_div(total_transitions, n_sentences)
    f["percent_sophisticated_transitions"] = safe_div(a["num_sophisticated_transitions"], total_transitions) * 100

    f["percent_sentences_w_one_concept"] = safe_div(a["num_one_concept_sentences"], n_sentences) * 100
    f["percent_sentences_w_multi_concept"] = safe_div(a["num_multi_concept_sentences"], n_sentences) * 100
    f["percent_cleft_sentences"] = safe_div(a["num_cleft_sentences"], n_sentences) * 100

    return f


FEATURE_COLS = [
    "avg_words_per_sentence", "sentence_length_variation", "percent_short_sentences",
    "percent_medium_sentences", "percent_long_sentences", "percent_very_long_sentences",
    "flesch_kincaid_grade", "percent_simple_sentences", "percent_compound_sentences",
    "percent_complex_sentences", "percent_compound_complex_sentences", "percent_other_sentences",
    "percent_words_in_simple_sentences", "percent_words_in_complex_sentences",
    "percent_words_in_compound_sentences", "percent_words_in_compound_complex_sentences",
    "percent_words_in_other_sentences", "avg_subordinates_per_sentence", "avg_clauses_per_sentence",
    "percent_sentences_with_subordinate", "percent_sentences_with_multiple_subordinates",
    "percent_sentences_with_embedded_clauses", "prep_phrase_density", "participle_phrase_density",
    "appositive_phrase_density", "avg_transitions_per_sentence", "percent_sophisticated_transitions",
    "percent_sentences_w_one_concept", "percent_sentences_w_multi_concept", "percent_cleft_sentences",
    "max_clauses_in_any_sentence", "num_sentences", "num_simple_sentences", "num_compound",
    "num_basic_complex", "num_advanced_complex", "percentage_simple", "percentage_compound",
    "percentage_basic_complex", "percentage_advanced_complex",
]


def features_to_json(features: dict, decimals: int = 1, cast_to_int: bool = True) -> str:
    payload = {}
    for key in FEATURE_COLS:
        value = features.get(key)
        if value is None:
            payload[key] = None
            continue
        value = round(value, decimals)
        payload[key] = int(value) if cast_to_int else value
    return json.dumps(payload, indent=2)

### Rubric selection via condition

Every `preprocessing` entry may declare a `condition` (`{"input": ..., "in": [...]}`), evaluated
generically against the actual input. `rubric_grade_3`/`rubric_grade_4`/`rubric_grades_5_12` all
share the output name `rubric` and have mutually exclusive conditions on `grade_level` — exactly one
always matches. This is the same generic condition mechanism Vocabulary Complexity uses for its two
`vocab_complexity_*` steps, applied here to `preprocessing` entries instead.

In [ ]:
_RUBRIC_FILES = {
    "rubric_grade_3": "rubric-grade-3.txt",
    "rubric_grade_4": "rubric-grade-4.txt",
    "rubric_grades_5_12": "rubric-grades-5-12.txt",
}


def condition_matches(condition, inputs):
    """A preprocessing/step entry with no condition always applies."""
    if condition is None:
        return True
    value = str(inputs.get(condition["input"]))
    return value in [str(v) for v in condition["in"]]


def get_rubric_for_grade(grade_level: str) -> str:
    candidates = [
        p for p in CONFIG["preprocessing"]
        if p["kind"] == "load_rubric_text" and condition_matches(p.get("condition"), {"grade_level": grade_level})
    ]
    if len(candidates) != 1:
        raise ValueError(f"expected exactly one matching rubric for grade_level={grade_level!r}, got {len(candidates)}")
    return (ASSETS_DIR / _RUBRIC_FILES[candidates[0]["id"]]).read_text()

### Stage 2: classify complexity

Structured output is driven by `output_schema.json` (`complexity_score`, `reasoning`).

In [ ]:
def run_classify_complexity(features_json: str, grade_level: str, excerpt: str) -> dict:
    step = STEPS["classify_complexity"]["spec"]
    llm = ChatOpenAI(model=step["model"]["name"], temperature=step["generation"]["temperature"])
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    rubric = get_rubric_for_grade(grade_level)
    prompt_template = ChatPromptTemplate.from_messages(STEPS["classify_complexity"]["messages"])
    rendered_messages = prompt_template.format_messages(
        grade_level=grade_level, rubric=rubric, excerpt=excerpt, sentence_features=features_json,
    )
    raw = structured_llm.invoke(rendered_messages)

    if raw.get("parsing_error"):
        raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

    return {
        "rendered_prompt": [m.model_dump() for m in rendered_messages],
        "raw_output": raw["raw"],
        "raw_text": raw["raw"].content,
        "formatted_output": raw["parsed"],
        "usage": getattr(raw["raw"], "usage_metadata", None),
    }

### Put it together

In [ ]:
def evaluate_sentence_structure(text: str, grade_level: str):
    """
    Evaluate sentence structure complexity for a text at a given grade, using the canonical
    config.json in this directory.

    Returns a dict with:
      - analysis:      stage 1's full I/O trace dict
      - features:      the engineered features dict
      - features_json: the JSON string fed into stage 2 as {sentence_features}
      - classification: stage 2's full I/O trace dict
    """
    try:
        analysis = run_sentence_analysis(text)
        features = add_engineered_features(analysis["formatted_output"])
        features_json = features_to_json(features)
        classification = run_classify_complexity(features_json, grade_level, text)
        return {
            "analysis": analysis,
            "features": features,
            "features_json": features_json,
            "classification": classification,
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

# Try evaluating text for sentence structure

Evaluator may take up to a minute to run (two model calls: sentence analysis, then complexity
classification). To evaluate your own text, replace the text and grade below and run the cell again.

In [ ]:
# Swap out text and grade here for your own text to evaluate
sample_text = "The cat sat on the mat. It was sleeping peacefully. The sun was warm. Soon it began to dream."

result = evaluate_sentence_structure(sample_text, grade_level="3")

if isinstance(result, dict):
    out = result["classification"]["formatted_output"]
    print(f"Complexity score: {out['complexity_score']}")
    print(f"\nReasoning:\n{out['reasoning']}")
else:
    print(result)

### Full I/O trace

In [ ]:
if isinstance(result, dict):
    print("=" * 60)
    print("STAGE 1: RENDERED PROMPT (sentence_analysis)")
    print("=" * 60)
    pp.pprint(result["analysis"]["rendered_prompt"])

    print("\n" + "=" * 60)
    print("STAGE 1: PARSED OUTPUT")
    print("=" * 60)
    pp.pprint(result["analysis"]["formatted_output"])

    print("\n" + "=" * 60)
    print("ENGINEERED FEATURES (fed into stage 2 as {sentence_features})")
    print("=" * 60)
    print(result["features_json"])

    print("\n" + "=" * 60)
    print("STAGE 2: RENDERED PROMPT (classify_complexity)")
    print("=" * 60)
    pp.pprint(result["classification"]["rendered_prompt"])

    print("\n" + "=" * 60)
    print("STAGE 2: PARSED OUTPUT")
    print("=" * 60)
    pp.pprint(result["classification"]["formatted_output"])
else:
    print(result)

### Sniff-test runner

Runs the cases in `fixtures.json` and compares the predicted complexity level against the
expected label. Per `config.json`'s `fixtures.tolerance`, a prediction one rubric step away
from the expected label also counts as a pass.

In [ ]:
fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

_RUBRIC_ORDER = OUTPUT_SCHEMA["properties"]["complexity_score"]["enum"]
_ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))

def _score_outcome(predicted: str, expected: str):
    """Return ('exact' | 'adjacent' | 'fail', distance_or_None)."""
    if predicted == expected:
        return "exact", 0
    if _ALLOW_ADJ and predicted in _RUBRIC_ORDER and expected in _RUBRIC_ORDER:
        d = abs(_RUBRIC_ORDER.index(predicted) - _RUBRIC_ORDER.index(expected))
        if d == 1:
            return "adjacent", d
    return "fail", None

results = []
for fx in fixtures:
    expected = fx["expected"]["complexity_score"]
    out = evaluate_sentence_structure(text=fx["input"]["text"], grade_level=fx["input"]["grade_level"])
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "predicted": None, "expected": expected, "error": out})
        continue
    predicted = out["classification"]["formatted_output"]["complexity_score"]
    status, _ = _score_outcome(predicted, expected)
    results.append({
        "id": fx["id"], "status": status,
        "predicted": predicted, "expected": expected,
        "description": fx.get("description", ""),
    })

print("=" * 78)
print(f"{'ID':>14}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
print("=" * 78)
for r in results:
    icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
    print(f"{r['id']:>14}  {icon:<8}  {(r['predicted'] or '-'):<22}  {r['expected']:<22}  {r.get('description','')[:25]}")

n_total = len(results)
n_exact = sum(1 for r in results if r["status"] == "exact")
n_adj   = sum(1 for r in results if r["status"] == "adjacent")
n_fail  = sum(1 for r in results if r["status"] == "fail")
n_err   = sum(1 for r in results if r["status"] == "error")
print("=" * 78)
print(f"Summary: {n_exact} exact, {n_adj} adjacent (tolerated), {n_fail} fail, {n_err} error  --  total {n_total}")
if _ALLOW_ADJ:
    print("(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)")

You can copy or edit the above cells to test out different texts and grade levels.